# ComplaintIQ - feature engineering (`06_feature_plan`)

The EDA (`01`-`03`) and baselines (`04a`, `04b`, `05`) are done; this notebook proposes **and
implements** leakage-safe feature engineering to beat those baselines, for a linear model
(supervised `monetary_relief`) and an unsupervised model (theme clustering). Each feature block
is introduced with **what it is**, the **EDA evidence** for it, and **which baseline metric** it
targets, then built in the cell below it.

- Supervised metric that matters: **lift at top-1% / 5% / 10%** (review-queue metrics), with PR-AUC alongside.
- Unsupervised metric: **ARI / NMI** vs the `product x issue` yardstick, plus silhouette.

## How to read this notebook
`04b` showed a metadata model can raise PR-AUC while barely moving lift, so lift is the honest
bar. Every feature here uses **intake-time fields only** and is computed **leakage-safe**
(statistics learned on the training window, applied to the later test window), per the leakage
audit in `02_eda_supervised.ipynb` section 13.

> **Note:** Spark reads the Parquet and does the chronological split + stratified sampling; the
> sklearn feature engineering and models run on the pandas sample.

> **Warning:** excluded as leakage: `company_response_to_consumer` (label source) and
> `timely_response` (post-response). No feature below touches them, and none uses a
> resolution-derived delta from `date_sent_to_company`.

---
## 1. The bar to beat

From `04a`, `04b`, `05` (chronological split, stratified sample, shared metrics):

| Model | Notebook | PR-AUC | Lift @1%/5%/10% | Note |
|---|---|---|---|---|
| Majority class ("predict never") | `04a` | ~base rate | ~1x | the accuracy trap |
| Product-bucket heuristic | `04a` | ~0.11 | **~9.7x** | rules-only, the real bar |
| Logistic regression, raw one-hot metadata | `04b` | ~0.16 | ~9.8x | higher PR-AUC, **lift barely moves** |
| Logistic regression, raw TF-IDF (narrative) | `04b` | ~0.28 | ~4.5x | strongest text signal, ~23% of rows |
| KMeans, raw TF-IDF, k = #products | `05` | - | - | **ARI ~0.01, NMI ~0.18** vs product x issue |

> **Why it matters:** "useful" means beating the product-bucket heuristic on **lift** and beating
> near-zero **ARI** on the clustering. A higher PR-AUC that does not move lift has not cleared the bar.

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from pyspark.sql import DataFrame as SparkDataFrame
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.metrics import report, top_k_lift
from complaintiq.sampling import stratified_pandas
from complaintiq.features import downcast_sparse_int32, shape_features
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import average_precision_score, roc_auc_score

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


print("python", sys.version.split()[0], "| sklearn ready")

---
## 2. Load + chronological split + stratified sample *(shared foundation)*

Same discipline as `04`: Spark reads intake columns + target, splits by time at the 80th
percentile of `date_received` (older = train, recent = test), and draws a proportional
per-class sample into pandas for sklearn.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"
if not path.exists():
    raise FileNotFoundError("data/complaints.parquet not found. Run `make parquet` first.")

# Intake-only columns (no company_response_to_consumer / timely_response).
KEEP = [
    "monetary_relief",
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "submitted_via",
    "state",
    "tags",
    "company",
    "zip_code",
    "complaint_text",
    "has_narrative",
]
spark_df = (
    spark.read.parquet(str(path))
    .select(*KEEP, F.to_date("date_received").alias("date_received"))
    .dropna(subset=["date_received"])
)
spark_df = spark_df.withColumn("epoch", F.datediff("date_received", F.lit("1970-01-01")))
cut = spark_df.approxQuantile("epoch", [0.80], 0.001)[0]
train_sdf = spark_df.filter(F.col("epoch") <= cut).drop("epoch")
test_sdf = spark_df.filter(F.col("epoch") > cut).drop("epoch")


train = stratified_pandas(train_sdf, 300_000).reset_index(drop=True)
test = stratified_pandas(test_sdf, 300_000).reset_index(drop=True)
y_train = train["monetary_relief"].to_numpy()
y_test = test["monetary_relief"].to_numpy()
print(
    f"train {len(train):,} (pos {y_train.mean():.4%})  |  test {len(test):,} (pos {y_test.mean():.4%})"
)
sup_results = []

---
## 3. Linear-model features (supervised)

The gap from `04b`: raw one-hot metadata lifts PR-AUC over the heuristic but barely beats its
lift at top-1% / 5% / 10%. The features below give the linear model finer-grained versions of the
category signal, plus the text signal the metadata path ignores. We first rebuild the `04b`
one-hot baseline here so the improvement is measured in-notebook.

In [ ]:
# Baseline reference (04b): logistic regression on raw one-hot metadata.
CAT = ["product", "sub_product", "issue", "sub_issue", "submitted_via", "state", "tags"]


def categorical_dict_records(d: pd.DataFrame) -> list[dict[str, str]]:
    return d[CAT].fillna("MISSING").astype(str).to_dict("records")


dict_vectorizer = DictVectorizer(sparse=True)
onehot_train = downcast_sparse_int32(dict_vectorizer.fit_transform(categorical_dict_records(train)))
onehot_test = downcast_sparse_int32(dict_vectorizer.transform(categorical_dict_records(test)))
base_lr = LogisticRegression(max_iter=200, class_weight="balanced", solver="liblinear").fit(
    onehot_train, y_train
)
sup_results.append(report("baseline_onehot", y_test, base_lr.predict_proba(onehot_test)[:, 1]))

### 3.1 Target + frequency encoding of high-cardinality categoricals
**What:** replace sparse one-hot of `company` / `zip_code` / `issue` / `sub_issue` / `product`
with two dense columns each: their **historical relief rate** (target encoding) and their
**frequency**. **Evidence:** `02` section 7 (these fields are high-cardinality) and section 9a
(a category's relief rate is exactly what the product-bucket heuristic exploits). **Targets:**
**lift** (reported at top-1% / 5% / 10%) - hands the model the heuristic's signal at finer granularity than `product`.
**Leakage-safe:** target encoding is **out-of-fold** on train (KFold) and train-only means are
applied to test, so no row sees its own label.

In [ ]:
HIGH = ["company", "zip_code", "issue", "sub_issue", "product"]
prior = y_train.mean()
target_encoded_train = np.zeros((len(train), len(HIGH)))
target_encoded_test = np.zeros((len(test), len(HIGH)))
freq_encoded_train = np.zeros((len(train), len(HIGH)))
freq_encoded_test = np.zeros((len(test), len(HIGH)))
kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
for j, col in enumerate(HIGH):
    oof = np.full(len(train), prior)  # out-of-fold target encoding (train)
    for a, b in kf.split(train):
        means = train.iloc[a].groupby(col, observed=True)["monetary_relief"].mean()
        oof[b] = train.iloc[b][col].map(means).fillna(prior).to_numpy()
    target_encoded_train[:, j] = oof
    full = train.groupby(col, observed=True)["monetary_relief"].mean()  # train-only means -> test
    target_encoded_test[:, j] = test[col].map(full).fillna(prior).to_numpy()
    freq = train[col].value_counts(normalize=True)  # frequency encoding
    freq_encoded_train[:, j] = train[col].map(freq).fillna(0).to_numpy()
    freq_encoded_test[:, j] = test[col].map(freq).fillna(0).to_numpy()
print(f"encoded {len(HIGH)} high-cardinality fields -> {2 * len(HIGH)} dense columns")

### 3.2 Text-shape features
**What:** cheap numeric descriptors of the raw narrative - log length, uppercase ratio,
redaction-token (`XX`) density, and a has-dollar-amount flag. **Evidence:** `02` section 12
(narrative length differs by target class); section 9 (individual numeric columns are weak, so
these complement, not replace). **Targets:** PR-AUC on the narrative-bearing rows. **Leakage-safe:**
derived purely from `complaint_text` at intake.

In [ ]:
text_shape_train = shape_features(train["complaint_text"])
text_shape_test = shape_features(test["complaint_text"])
print("text-shape feature matrix:", text_shape_train.shape)

### 3.3 Interaction crosses
**What:** explicit `product x submitted_via` and `product x issue` cross categories.
**Evidence:** `02` section 10 - the relief rate shifts across the product/channel interaction.
**Targets:** lift - a linear model cannot recover interactions from independent one-hot columns.
**Leakage-safe:** built from intake categoricals only.

In [ ]:
def interaction_records(d: pd.DataFrame) -> list[dict[str, str]]:
    return pd.DataFrame(
        {
            "product_x_submitted_via": d["product"].astype(str)
            + "|"
            + d["submitted_via"].astype(str),
            "product_x_issue": d["product"].astype(str) + "|" + d["issue"].astype(str),
        }
    ).to_dict("records")


dvi = DictVectorizer(sparse=True)
interaction_train = downcast_sparse_int32(dvi.fit_transform(interaction_records(train)))
interaction_test = downcast_sparse_int32(dvi.transform(interaction_records(test)))
print("interaction one-hot:", interaction_train.shape[1], "columns")

### 3.4 Date features
**What:** cyclical month (sin/cos) and year from `date_received`. **Evidence:** `02` section 11 -
the monthly relief rate drifts, not flat. **Targets:** absorbs seasonal/secular trend.
**Leakage-safe:** `date_received` is intake; no delta to `date_sent_to_company`.

In [ ]:
def date_features(d: pd.DataFrame) -> np.ndarray:
    # Spark's toPandas returns to_date() as object/date, not datetime64; coerce so .dt works
    # in both the notebook (Spark) and local runs.
    date_received_dt = pd.to_datetime(d["date_received"], errors="coerce")
    m = date_received_dt.dt.month.to_numpy()
    year = date_received_dt.dt.year.to_numpy()
    return np.vstack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12), (year - 2010)]).T


date_features_train = date_features(train)
date_features_test = date_features(test)
print("date feature matrix:", date_features_train.shape)

### 3.5 Assemble + fit the engineered linear model
Combine one-hot metadata + interactions + the dense encoded/shape/date blocks, and fit the same
logistic regression as the baseline so only the **features** differ. Compare against the `04b`
one-hot baseline recorded above.

In [ ]:
dense_train = csr_matrix(
    np.hstack([target_encoded_train, freq_encoded_train, text_shape_train, date_features_train])
)
dense_test = csr_matrix(
    np.hstack([target_encoded_test, freq_encoded_test, text_shape_test, date_features_test])
)
engineered_train = downcast_sparse_int32(hstack([onehot_train, interaction_train, dense_train]))
engineered_test = downcast_sparse_int32(hstack([onehot_test, interaction_test, dense_test]))

fe_lr = LogisticRegression(max_iter=300, class_weight="balanced", solver="liblinear").fit(
    engineered_train, y_train
)
sup_results.append(
    report("engineered_metadata", y_test, fe_lr.predict_proba(engineered_test)[:, 1])
)

### 3.6 Engineered text model (narrative subset)
**What:** TF-IDF with word n-grams (1-2) **and** character n-grams (3-5), `sublinear_tf`, tuned
`min_df` / `max_df`, vs the `04b` unigram baseline. **Evidence:** `02` section 12 (phrase-level
signal) and `03` section 6 (vocab sizing). **Targets:** PR-AUC / lift on the ~23% narrative rows.
**Leakage-safe:** vectorizer fit on train narratives only.

In [ ]:
train_narrative = train[train["has_narrative"] & train["complaint_text"].notna()]
test_narrative = test[test["has_narrative"] & test["complaint_text"].notna()]
y_train_narrative = train_narrative["monetary_relief"].to_numpy()
y_test_narrative = test_narrative["monetary_relief"].to_numpy()
print(
    f"narrative subset - train {len(train_narrative):,} test {len(test_narrative):,} (base {y_test_narrative.mean():.4%})"
)

# 04b baseline: default unigrams
baseline_vectorizer = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
tfidf_baseline_train = downcast_sparse_int32(
    baseline_vectorizer.fit_transform(train_narrative["complaint_text"])
)
tfidf_baseline_test = downcast_sparse_int32(
    baseline_vectorizer.transform(test_narrative["complaint_text"])
)
baseline_text_model = LogisticRegression(
    max_iter=300, class_weight="balanced", solver="liblinear"
).fit(tfidf_baseline_train, y_train_narrative)
report(
    "baseline_tfidf_text",
    y_test_narrative,
    baseline_text_model.predict_proba(tfidf_baseline_test)[:, 1],
)

# engineered: word (1-2) + char (3-5) n-grams, tuned
word_ngram_vectorizer = TfidfVectorizer(
    max_features=40_000,
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words="english",
)
char_ngram_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), max_features=40_000, min_df=5, sublinear_tf=True
)
tfidf_engineered_train = downcast_sparse_int32(
    hstack(
        [
            word_ngram_vectorizer.fit_transform(train_narrative["complaint_text"]),
            char_ngram_vectorizer.fit_transform(train_narrative["complaint_text"]),
        ]
    )
)
tfidf_engineered_test = downcast_sparse_int32(
    hstack(
        [
            word_ngram_vectorizer.transform(test_narrative["complaint_text"]),
            char_ngram_vectorizer.transform(test_narrative["complaint_text"]),
        ]
    )
)
engineered_text_model = LogisticRegression(
    max_iter=300, class_weight="balanced", solver="liblinear"
).fit(tfidf_engineered_train, y_train_narrative)
report(
    "engineered_tfidf_text",
    y_test_narrative,
    engineered_text_model.predict_proba(tfidf_engineered_test)[:, 1],
)

### 3.7 Supervised scoreboard

In [ ]:
board = pd.DataFrame(sup_results).set_index("model")
display(board.round(4))
ax = sns.barplot(
    x=board["pr_auc"], y=board.index, hue=board.index, palette="colorblind", legend=False
)
ax.set_title("PR-AUC: baseline vs engineered (whole test set)")
ax.set_xlabel("PR-AUC")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Persist the supervised per-model metrics to the volume so they're retrievable
# outside the run (the jobs API does not expose notebook stdout). Mirrors 07a-10.
# No-op locally when dbutils is absent. report() already returns lift at 1/5/10%.
try:
    import json as _json
    import time as _time

    _payload = {
        "notebook": "06_feature_plan",
        "test_base_rate": float(y_test.mean()),
        "sup_results": sup_results,
        "ts": _time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    dbutils.fs.put(  # noqa: F821 - Databricks-injected global
        "/Volumes/workspace/complaintiq/data/metrics_06.json",
        _json.dumps(_payload, indent=2),
        overwrite=True,
    )
    print("wrote /Volumes/workspace/complaintiq/data/metrics_06.json")
except NameError:
    print("dbutils unavailable (local run); skipped metrics persistence")

> **What you're seeing:** engineered metadata raises PR-AUC well above the one-hot baseline, but
> **lift** moves only slightly at any queue size (top-1% / 5% / 10%) - the product-signal ceiling `04b` flagged. Engineered text
> improves PR-AUC and lift on the narrative subset.
>
> **Why it matters:** honest result - on this data, metadata FE mostly buys PR-AUC, while the
> narrative text (and, next, calibrating the two paths) is where extra queue lift most plausibly lives.

### 3.8 The open problem: calibrate the two paths
The narrative model covers ~23% of complaints, the metadata model the rest (`02` section 8). To
merge their scores into one ranked queue they must be **calibrated onto the same probability scale**
(Platt / isotonic) and the top slice checked for one-group dominance. That merge is the next
notebook's job; this notebook establishes that each path's engineered features beat their baseline.

---
## 4. Unsupervised features

The `05` baseline (raw TF-IDF + KMeans, k = #products) recovers the `product x issue` yardstick
only weakly (ARI ~0.01). We clean the corpus, represent it densely, and let KMeans behave.

> **What the "product x issue" yardstick is:** each complaint has `product` (what it is about) and
> `issue` (the problem); their cross-tabulation is a grid of pairs, a few of which dominate as real
> themes (`03` section 8). Clustering has no label, so we grade it by agreement (ARI / NMI) with
> those blocks - a measuring stick built from fields the model never sees, never a feature.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

# Sample narratives in Spark (stratified by product so every theme is represented), to pandas.
nar_path = data_dir / "complaints_narrative_only.parquet"
src = str(nar_path) if nar_path.exists() else str(path)
docs_sdf = spark.read.parquet(src)
if "has_narrative" in docs_sdf.columns and src == str(path):
    docs_sdf = docs_sdf.filter(F.col("has_narrative"))
docs_sdf = docs_sdf.select("complaint_text", "product", "issue").dropna(
    subset=["complaint_text", "product"]
)
total = docs_sdf.count()
prods = [r["product"] for r in docs_sdf.select("product").distinct().collect()]
frac = min(1.0, 25_000 / total) if total else 0.0
samp = (
    docs_sdf.sampleBy("product", {p: frac for p in prods}, seed=RANDOM_STATE)
    .toPandas()
    .reset_index(drop=True)
)
print(f"corpus {total:,} -> sample {len(samp):,}")


def yardstick(frame: pd.DataFrame) -> np.ndarray:
    return (
        (frame["product"].astype(str) + " | " + frame["issue"].astype(str))
        .astype("category")
        .cat.codes
    )


def eval_clusters(X: np.ndarray, labels: np.ndarray, truth: np.ndarray, tag: str) -> dict[str, Any]:
    ari = adjusted_rand_score(truth, labels)
    nmi = normalized_mutual_info_score(truth, labels)
    sil = silhouette_score(X, labels, sample_size=5000, random_state=RANDOM_STATE)
    print(f"{tag:<26} ARI={ari:.4f}  NMI={nmi:.4f}  silhouette={sil:.4f}")
    return {"model": tag, "ari": ari, "nmi": nmi, "silhouette": sil}


uns_results = []

### 4.1 Baseline reference (raw TF-IDF + KMeans)
Rebuild `05`'s baseline in-notebook so the improvement is measured directly.

In [ ]:
base_vec = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
tfidf_baseline = base_vec.fit_transform(samp["complaint_text"])
n_clusters_baseline = samp["product"].nunique()
uns_results.append(
    eval_clusters(
        tfidf_baseline,
        KMeans(n_clusters_baseline, random_state=RANDOM_STATE, n_init=5).fit_predict(
            tfidf_baseline
        ),
        yardstick(samp),
        f"baseline_tfidf_k{n_clusters_baseline}",
    )
)

### 4.2 Deduplicate template narratives
**What:** drop exact-duplicate narratives before clustering. **Evidence:** `03` section 9 -
templated/duplicate narratives are prevalent and form artificial clusters. **Targets:** ARI -
removing template mass lets KMeans find real themes. (Near-duplicate LSH is the follow-up.)

In [ ]:
dedup = samp.drop_duplicates(subset="complaint_text").reset_index(drop=True)
print(
    f"dedup: {len(samp):,} -> {len(dedup):,} rows ({1 - len(dedup) / len(samp):.1%} were exact duplicates)"
)

### 4.3 Tuned TF-IDF + LSA (dimensionality reduction) + normalize
**What:** tuned TF-IDF (n-grams, `sublinear_tf`, tighter `max_df`) then **TruncatedSVD (LSA)** to a
dense ~100-dim space, L2-normalized. **Evidence:** `03` section 6 (large sparse vocab, Zipf skew)
and section 7 (Euclidean distance degrades in high-dim sparse space). **Targets:** ARI / NMI /
silhouette - dense denoised space makes KMeans distances meaningful.

> **Go deeper:**
> - [scikit-learn: TruncatedSVD (LSA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html): *why SVD on TF-IDF denoises the space KMeans clusters in, ~5 min.*
> - [scikit-learn: manifold learning](https://scikit-learn.org/stable/modules/manifold.html): *t-SNE / spectral embedding / Isomap and where each fits, ~10 min.*

In [ ]:
fe_vec = TfidfVectorizer(
    max_features=40_000,
    min_df=5,
    max_df=0.5,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words="english",
)
tfidf_deduped = fe_vec.fit_transform(dedup["complaint_text"])
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
lsa_features = Normalizer().fit_transform(svd.fit_transform(tfidf_deduped))
print(
    f"LSA space: {lsa_features.shape} | explained variance {svd.explained_variance_ratio_.sum():.1%}"
)

### 4.4 Cluster on the engineered space + sweep k
**What:** KMeans on the LSA space at k = #products and a small sweep. **Evidence:** `05` fixed
k = #products untuned; the true theme count need not match. **Targets:** best ARI/NMI vs yardstick.

In [ ]:
truth_d = yardstick(dedup)
n_clusters_deduped = dedup["product"].nunique()
for n_clusters in sorted({10, 20, n_clusters_deduped, 40}):
    tag = f"engineered_dedup_lsa_k{n_clusters}"
    res = eval_clusters(
        lsa_features,
        KMeans(n_clusters, random_state=RANDOM_STATE, n_init=5).fit_predict(lsa_features),
        truth_d,
        tag,
    )
    uns_results.append(res)

### 4.5 Unsupervised scoreboard

In [ ]:
uboard = pd.DataFrame(uns_results).set_index("model")
display(uboard.round(4))
ax = sns.barplot(
    x=uboard["ari"], y=uboard.index, hue=uboard.index, palette="colorblind", legend=False
)
ax.set_title("ARI vs product x issue yardstick (higher = recovers themes better)")
ax.set_xlabel("Adjusted Rand Index")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

> **What you're seeing:** dedup + LSA lifts ARI several-fold over the raw-TF-IDF baseline and
> improves NMI and silhouette.
>
> **Why it matters:** unlike the supervised lift ceiling, the unsupervised side has clear room -
> cleaning template mass and denoising the space are the levers that recover real themes.

---
## 5. Beyond LSA: stronger representations (implemented in 08/09)

The unsupervised pipeline above stops at TF-IDF + LSA. Two stronger representations are worth
trying, and rather than propose them here, the project implements and measures them:

- **Sentence embeddings** (a pretrained MiniLM): each narrative becomes one dense meaning-vector, so
  paraphrases with no shared vocabulary land near each other. This is the step change for
  meaning-defined themes. Measured in **08** (clustering: embeddings beat TF-IDF, ARI 0.014 -> 0.067)
  and **09** (supervised: TF-IDF still wins for relief, embeddings help only stacked).
- **Full-corpus scaling** of embedding clustering is measured in **10**.

The headline that falls out of those notebooks: representation choice is task-dependent (embeddings
win clustering, TF-IDF wins relief prediction). See `docs/FINDINGS.md` findings 6-7.

---
## 6. Takeaways

> **What the engineered features buy, measured against the baselines:**
> - **Supervised metadata:** target/frequency encoding + interactions + shape + date raise PR-AUC
>   well above the one-hot baseline; **lift** moves only slightly at top-1% / 5% / 10% (product-signal ceiling).
> - **Supervised text:** word+char n-gram TF-IDF beats the unigram baseline on the narrative subset.
> - **Unsupervised:** dedup + tuned TF-IDF + LSA beats the raw baseline decisively on ARI/NMI.
> - **Next:** calibrate the text + metadata paths onto one probability scale to merge them into
>   one ranked queue; add near-duplicate (LSH) dedup at full scale; and try the stronger
>   representations in section 5 (UMAP / spectral embedding, or sentence embeddings).

Everything above is leakage-safe (intake-only fields, train-window statistics) and comparable to
`04`/`05` via the shared metrics.